# 01 — Data Loading | Retail Sales Analytics

**Goal:** confirm that the Kaggle retail CSV files are present, load them into Spark DataFrames, and inspect each dataset's schema, sample rows, and row count. No cleaning, transformation, SQL analysis, or Delta writes are performed here.

**Source:** [Kaggle — Store Sales: Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data). Download and upload the original ZIP (or CSV files) into a Databricks Unity Catalog Volume before running.

**Environment:** Databricks Notebook (Python, Serverless). The `spark` session is available automatically. Set the Volume path below to match your workspace if necessary.

## 1. Check raw files and extract only if needed

First, check the Volume. If any expected CSV is missing and the ZIP exists, extract **only the missing CSV files**. Already-extracted files are left untouched, so rerunning the cell does not repeatedly unzip everything.

In [0]:
import os
import zipfile

RAW_VOLUME = "/Volumes/workspace/default/retail_raw"
ZIP_NAME = "store-sales-time-series-forecasting.zip"
EXPECTED_FILES = [
    "train.csv", "stores.csv", "transactions.csv",
    "holidays_events.csv", "oil.csv", "test.csv", "sample_submission.csv"
]

missing_files = [
    filename for filename in EXPECTED_FILES
    if not os.path.isfile(f"{RAW_VOLUME}/{filename}")
]

if missing_files:
    zip_path = f"{RAW_VOLUME}/{ZIP_NAME}"
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"Missing files {missing_files}; ZIP not found: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as archive:
        for filename in missing_files:
            archive.extract(filename, RAW_VOLUME)
    print("Extracted:", ", ".join(missing_files))
else:
    print("All seven CSV files are already available. No extraction needed.")

print("CSV files in Volume:")
for filename in sorted(EXPECTED_FILES):
    print(" -", filename)

## 2. Define one reusable CSV reader

All five datasets use the same CSV read options. Define a small helper **once** to avoid repeating the same `spark.read` chain. This helper only reads a file; it does not change the original CSV.

In [0]:
def read_raw_csv(filename):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{RAW_VOLUME}/{filename}")
    )

## 3. Sales history — `train.csv`

**Grain:** one date × store × product family. `sales` measures sales quantities, not revenue. Inspect the original six columns before any feature engineering.

In [0]:
df_sales = read_raw_csv("train.csv")

df_sales.printSchema()
print("Sales rows:", df_sales.count())
print("Sales columns:", len(df_sales.columns))
df_sales.show(5, truncate=False)

## 4. Store reference — `stores.csv`

**Grain:** one row per store, with city, state, type and cluster information. The join key is `store_nbr`.

In [0]:
df_stores = read_raw_csv("stores.csv")

df_stores.printSchema()
print("Store rows:", df_stores.count())
df_stores.show(5, truncate=False)

## 5. Transactions — `transactions.csv`

**Grain:** one date × store. `transactions` is the number of transactions, not sales quantity; joining it to the family-level sales table can multiply transaction counts unless data is aggregated to the matching grain.

In [0]:
df_transactions = read_raw_csv("transactions.csv")

df_transactions.printSchema()
print("Transaction rows:", df_transactions.count())
df_transactions.show(5, truncate=False)

## 6. Holidays and events — `holidays_events.csv`

Includes nationwide, regional and local events. The same date can have multiple rows and different `locale` values; a date-only join would risk duplicating sales rows. Detailed holiday cleaning is deferred until the analysis needs this table.

In [0]:
df_holidays = read_raw_csv("holidays_events.csv")

df_holidays.printSchema()
print("Holiday/event rows:", df_holidays.count())
df_holidays.show(5, truncate=False)

## 7. Oil prices — `oil.csv`

One date and WTI price value (`dcoilwtico`) per record. Some reported dates have NULL prices; those are handled in `02_Data_Cleaning`.

In [0]:
df_oil = read_raw_csv("oil.csv")

df_oil.printSchema()
print("Oil rows:", df_oil.count())
df_oil.show(10, truncate=False)

## Summary and next step

The original CSVs have been inspected; no source files were altered. `test.csv` and `sample_submission.csv` remain in the Volume for the later Kaggle forecasting phase.

Proceed to **`02_Data_Cleaning`** for NULL, key-uniqueness, range and date checks, transformations and managed Delta Tables.

**Note:** The yearly sales aggregation and average-daily-sales exercise from the original notebook belong to the planned `03_Sales_Analysis`. They are intentionally omitted here, not treated as data cleaning.